# DENTEX 4-Class Pathology Detection - Local Interactive Benchmark

This interactive notebook is designed for local workstations (e.g. NVIDIA RTX 3050 4GB VRAM):
- Probes local GPU and demonstrates memory-safe defaults (FP16, gradient accumulation)
- Matplotlib visualizer color-coded by the 4 target pathology classes:
  1. `Caries` (#E69F00 Orange)
  2. `Deep Caries` (#D55E00 Vermilion)
  3. `Periapical Lesion` (#0072B2 Blue)
  4. `Impacted Tooth` (#009E73 Green)
- Interactive checkpoint loading and inference on dental radiographs


## 1. System Diagnostics and Memory-Safe Defaults


In [ ]:
import os
import sys
from pathlib import Path
import torch

# Ensure project root is in Python module search path
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
    os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dentex_bench.env import detect_platform, get_device_info
from dentex_bench.config import BenchmarkConfig
from dentex_bench.data.parser import TARGET_CLASSES

dev = get_device_info()
print(f'Platform: {detect_platform()}')
print(f'GPU: {dev["device"]} ({dev["name"]}), VRAM: {dev["vram_gb"]} GB, Low VRAM: {dev["is_low_vram"]}')

cfg = BenchmarkConfig.from_auto_env()
print(f'Adaptive Config -> Batch size: {cfg.batch_size}, Grad Accum: {cfg.gradient_accumulation_steps}, Workers: {cfg.num_workers}')
print(f'Target classes: {TARGET_CLASSES}')


## 2. Interactive Bounding Box Visualizer for 4 Pathology Classes


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from typing import List, Dict, Any, Optional

# Clinical color palette for 4 DENTEX pathology classes
CLASS_METADATA = {
    1: {"name": "Caries", "color": "#E69F00"},             # Orange
    2: {"name": "Deep Caries", "color": "#D55E00"},        # Vermilion
    3: {"name": "Periapical Lesion", "color": "#0072B2"}, # Blue
    4: {"name": "Impacted Tooth", "color": "#009E73"},    # Green
}

def visualize_dentex_detections(
    image: np.ndarray,
    boxes: List[List[float]],
    category_ids: List[int],
    scores: Optional[List[float]] = None,
    title: str = 'DENTEX Dental Pathology Detections',
    figsize: tuple = (12, 6)
):
    """
    Visualizes panoramic dental X-ray with color-coded bounding boxes for:
    - Caries
    - Deep Caries
    - Periapical Lesion
    - Impacted Tooth
    """
    fig, ax = plt.subplots(figsize=figsize)
    if len(image.shape) == 2:
        ax.imshow(image, cmap='gray')
    else:
        ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    
    for i, (box, cat_id) in enumerate(zip(boxes, category_ids)):
        meta = CLASS_METADATA.get(cat_id, {"name": f"Class {cat_id}", "color": "#FFFF00"})
        color = meta["color"]
        name = meta["name"]
        score_text = f' ({scores[i]:.2f})' if scores and i < len(scores) else ''
        x, y, w, h = box
        
        rect = patches.Rectangle(
            (x, y), w, h,
            linewidth=2.5,
            edgecolor=color,
            facecolor='none',
            linestyle='-'
        )
        ax.add_patch(rect)
        ax.text(
            x, max(0, y - 6),
            f'{name}{score_text}',
            color='white',
            fontsize=9,
            fontweight='bold',
            bbox=dict(boxstyle='square,pad=0.2', facecolor=color, alpha=0.85, edgecolor='none')
        )
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


## 3. Test Visualizer on Mock Dental Panoramic Radiograph


In [ ]:
# Generate synthetic dental panoramic radiograph canvas
mock_img = np.ones((600, 1000, 3), dtype=np.uint8) * 40
for y in range(600):
    mock_img[y, :] += int(35 * np.sin(np.pi * y / 600))

sample_boxes = [
    [140, 220, 55, 65],   # Caries
    [310, 240, 75, 85],   # Deep Caries
    [530, 360, 70, 70],   # Periapical Lesion
    [820, 270, 85, 105],  # Impacted Tooth
]
sample_cats = [1, 2, 3, 4]
sample_scores = [0.95, 0.89, 0.92, 0.98]

visualize_dentex_detections(
    mock_img,
    sample_boxes,
    sample_cats,
    scores=sample_scores,
    title='Interactive Preview: 4-Class Dental Pathology Detection'
)


## 4. Load Detector Runner and Checkpoint


In [ ]:
from dentex_bench.models.faster_rcnn import FasterRCNNRunner
from dentex_bench.models.dino import DINORunner
from dentex_bench.models.hierarchical_det import HierarchicalDetRunner

# Select model: 'faster_rcnn', 'dino', or 'hierarchical_det'
model_type = 'faster_rcnn'
runner_cls = {'faster_rcnn': FasterRCNNRunner, 'dino': DINORunner, 'hierarchical_det': HierarchicalDetRunner}[model_type]
runner = runner_cls(cfg)
runner.build_model()

checkpoint_path = Path(f'outputs/{model_type}/best.pth')
if not checkpoint_path.exists():
    checkpoint_path = Path(f'outputs/{model_type}_best.pth')
if checkpoint_path.exists():
    runner.load_checkpoint(checkpoint_path)
    print(f'Successfully loaded checkpoint: {checkpoint_path}')
else:
    print(f'Checkpoint {checkpoint_path} not found. Running with initial model weights.')


## 5. Run Local Inference and Visual Inspection


In [ ]:
# Run prediction on image
sample_images = list(Path("data").glob("**/*.png")) + list(Path("data").glob("**/*.jpg"))
test_img_path = sample_images[0] if sample_images else Path("data/raw/sample_xray.png")
if test_img_path.exists():
    bgr_img = cv2.imread(str(test_img_path))
    img_rgb = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
    img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).float() / 255.0
    predictions = runner.predict([img_tensor])
    pred = predictions[0]

    keep = pred['scores'] >= 0.5
    boxes_xyxy = pred['boxes'][keep].cpu().numpy()
    boxes_xywh = [[float(x1), float(y1), float(x2 - x1), float(y2 - y1)] for x1, y1, x2, y2 in boxes_xyxy]
    labels = pred['labels'][keep].cpu().numpy().tolist()
    scores = pred['scores'][keep].cpu().numpy().tolist()

    visualize_dentex_detections(
        bgr_img,
        boxes_xywh,
        labels,
        scores,
        title=f'DENTEX Pathology Predictions ({model_type})'
    )
else:
    print(f'No image found at {test_img_path}.')
    print('Run scripts/download_and_prepare.py to populate test images.')
